# ChibiCreate — Notebook 1: modelo SOZINHO

```
PERSONAGEM ORIGINAL  →  MODELO SELECIONADO  →  OUTPUT
```

Descobre qual modelo resolve nosso problema **sem depender do FLUX**.
O eixo principal é **DESIGN_PRESERVATION**.

Escolha um modelo no dropdown, execute as células em ordem, analise, baixe o
ZIP, resete o runtime e escolha outro. **Não altere código** — tudo vem de
`config/model_eval_registry.yaml`.

## Modelos no dropdown

| Modelo | Refs | Licença | Comercial |
|---|---|---|---|
| LongCat-Image-Edit | 0 (1 imagem) | Apache-2.0 | ✅ verified |
| Z-Image Turbo | 0 (img2img) | Apache-2.0 | ✅ verified |
| Qwen-Image-Edit-2511 Q3_K_M | 2 | Apache-2.0 base | ⚠️ pending review |
| Qwen-Image-Edit-2511 Q4_0 | 2 | Apache-2.0 base | ⚠️ pending review |
| Pony Diffusion V6 XL | 0 (img2img) | FAIPL-1.0-SD mod. | ⛔ **research_only** |

⛔ **Pony: "Research only — not approved for commercial production".** Pode
ser testado tecnicamente; **não** entra no ranking comercial.

⚠️ **Qwen Q3/Q4 são quantizações de terceiro** (unsloth) e exigem o custom
node `ComfyUI-GGUF`. O notebook pede aceite explícito antes de instalar.

⚠️ **Z-Image Turbo é text-to-image**, usado aqui em img2img. Não é editor
multi-reference — a limitação fica registrada no recipe.

**Nada é baixado até você escolher e confirmar.**


## 1 — Setup: clonar, importar e validar o registry

In [ ]:
# SETUP — clone, sys.path, import do registry e VALIDACAO.
# Auto-suficiente: nao depende de nenhuma execucao anterior nem de celula
# opcional. Em runtime novo, esta e a PRIMEIRA celula a rodar.
import os, subprocess, sys, importlib

REPO_DIR = '/content/ChibiCreate'
SCRIPTS_DIR = REPO_DIR + '/scripts'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch',
                    'arena/01a07ece-chibicreate',
                    'https://github.com/BloomRX/ChibiCreate.git', REPO_DIR],
                   check=True)

os.chdir(REPO_DIR)
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyyaml', 'pillow', 'huggingface_hub'], check=True)

# O sintoma relatado foi 'NameError: name mr is not defined' no preflight.
# A causa real era esta importacao falhar (ou a celula ser pulada) e o erro
# so aparecer tres celulas depois, longe da origem. Agora falha aqui, com
# diagnostico e acao.
try:
    from chibi import model_registry as mr
    importlib.reload(mr)
except Exception as exc:
    raise RuntimeError(
        'FALHA AO IMPORTAR scripts/chibi/model_registry.py\n'
        f'  erro    : {type(exc).__name__}: {exc}\n'
        f'  cwd     : {os.getcwd()}\n'
        f'  arquivo : {os.path.isfile(SCRIPTS_DIR + "/chibi/model_registry.py")}\n'
        f'  sys.path: {sys.path[0]}\n'
        'Acao: Runtime > Restart session e rode esta celula primeiro.'
    ) from exc

FALTANDO = [n for n in ('preflight', 'load_registry', 'dropdown_options',
                        'key_for_label', 'get_model', 'prompt_for',
                        'plan_references', 'run_dir_for', 'describe',
                        'needs_confirmation', 'comparison_table')
            if not hasattr(mr, n)]
if FALTANDO:
    raise RuntimeError(
        'model_registry importado mas incompleto. Faltam: '
        + ', '.join(FALTANDO)
        + '\nRepo provavelmente desatualizado: apague /content/ChibiCreate '
          'e rode esta celula de novo.')

REG = mr.load_registry()
SETUP_OK = True

print('SETUP OK')
print('  repo        :', REPO_DIR)
print('  registry    :', len(REG['models']), 'modelos')
print('  mr.preflight:', callable(mr.preflight))
print()
print('Proxima celula: dropdown de modelos.')


## 2 — Dropdown de modelos

In [ ]:
# DROPDOWN. Depende apenas da celula de SETUP.
if not globals().get('SETUP_OK'):
    raise RuntimeError('Rode a celula 1 (SETUP) antes desta.')

OPCOES = mr.dropdown_options(REG)

print('-' * 64)
print('MODEL EVALUATION')
print('-' * 64)

USE_WIDGET = True
try:
    import ipywidgets as widgets
    from IPython.display import display
    MODEL_DROPDOWN = widgets.Dropdown(
        options=OPCOES, description='Model:',
        layout=widgets.Layout(width='560px'))
    display(MODEL_DROPDOWN)
except Exception as exc:
    # Widgets nem sempre sobem num runtime recem-criado. Sem fallback o
    # notebook morreria por um detalhe de UI.
    USE_WIDGET = False
    MODEL_DROPDOWN = None
    print('(ipywidgets indisponivel:', type(exc).__name__,
          '- selecione por indice na proxima celula)')

print()
for i, o in enumerate(OPCOES):
    print(f'  [{i}] {o}')
print()
print('Input mode: automatico (definido pelo registry)')
print()
print('Escolha na proxima celula. Nada e baixado ate confirmar.')


## 3 — Selecionar o modelo

In [ ]:
# SELECAO DO MODELO.
# MODEL_INDEX existe para 'Runtime > Run all': o dropdown grafico exige um
# clique humano, que Run All nao da. Se voce mexeu no dropdown, ele tem
# prioridade; caso contrario vale o indice abaixo.
MODEL_INDEX = 2          # 2 = Qwen-Image-Edit-2511 Q3_K_M

if not globals().get('SETUP_OK'):
    raise RuntimeError('Rode a celula 1 (SETUP) antes desta.')

if USE_WIDGET and MODEL_DROPDOWN is not None:
    MODEL_LABEL = MODEL_DROPDOWN.value
else:
    MODEL_LABEL = OPCOES[MODEL_INDEX]

MODEL_KEY = mr.key_for_label(MODEL_LABEL, REG)
CFG = mr.get_model(MODEL_KEY, REG)
PARAMS = CFG['parameters']

PROMPT_INFO = mr.prompt_for(MODEL_KEY, REG)
PROMPT = PROMPT_INFO['prompt']
NEGATIVE_PROMPT = PROMPT_INFO['negative_prompt']
SEED = 42
SELECTION_OK = True

print(mr.describe(MODEL_KEY, REG))
print()
print('PROMPT:', len(PROMPT), 'caracteres')
if PROMPT_INFO['override_applied']:
    print('  override do modelo:', PROMPT_INFO['override_reason'])
print('NEGATIVE PROMPT:', repr(NEGATIVE_PROMPT), '(sem negativas automaticas)')
print('SEED:', SEED)


## 4 — Referências da personagem

Faça upload do ZIP com as referências, ou use as que já vêm no repositório.
O notebook localiza `full_body.png`, `face.png` e `outfit.png`.

In [ ]:
import pathlib, hashlib
from PIL import Image
from IPython.display import display

REF_DIR = pathlib.Path('characters/waifu_001/reference')

USE_UPLOAD = False   # True para enviar um ZIP com as referencias
if USE_UPLOAD:
    from google.colab import files
    import zipfile, shutil
    dest = pathlib.Path('/content/refs')
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    up = files.upload()
    with zipfile.ZipFile(list(up)[0]) as z:
        z.extractall(dest)
    achados = {p.name: p for p in dest.rglob('*.png')}
    if 'full_body.png' not in achados:
        raise SystemExit('PARE: full_body.png nao encontrado no ZIP.')
    REF_DIR = achados['full_body.png'].parent

def ficha(p):
    data = p.read_bytes()
    with Image.open(p) as im:
        px = hashlib.sha256(im.convert('RGBA').tobytes()).hexdigest()
        wh = im.size
    return {'file': p.name, 'path': str(p),
            'artifact_sha256': hashlib.sha256(data).hexdigest(),
            'pixel_sha256': px, 'width': wh[0], 'height': wh[1],
            'bytes': len(data)}

DISPONIVEIS = {}
for nome in ('full_body.png', 'face.png', 'outfit.png'):
    p = REF_DIR / nome
    if p.is_file():
        DISPONIVEIS[nome] = ficha(p)
        print(f"  {nome:16} {DISPONIVEIS[nome]['width']}x{DISPONIVEIS[nome]['height']}"
              f"  pixel {DISPONIVEIS[nome]['pixel_sha256'][:16]}...")
    else:
        print(f'  {nome:16} AUSENTE')

if 'full_body.png' not in DISPONIVEIS:
    raise SystemExit('PARE: full_body.png e obrigatorio.')

display(Image.open(REF_DIR / 'full_body.png').resize((256, 256)))


### 4.1 — O que o modelo escolhido realmente aceita

Referência que o modelo não comporta **não é descartada em silêncio**:
aparece aqui e vai para o recipe.

In [ ]:
# A imagem principal e a propria personagem original.
PRIMARY = str(REF_DIR / 'full_body.png')
PRIMARY_ROLE = 'full_body'

DESEJADAS = [n for n in ('face.png', 'outfit.png') if n in DISPONIVEIS]

PLANO = mr.plan_references(MODEL_KEY, PRIMARY, PRIMARY_ROLE, DESEJADAS,
                           registry=REG)
print(PLANO.report())

if PLANO.has_dropped:
    print()
    print('Isto muda a leitura do resultado: este modelo trabalha com menos')
    print('informacao de design que os que aceitam referencias.')


## 5 — Preflight

In [ ]:
# PREFLIGHT — roda ANTES de qualquer download pesado.
# Valida as proprias dependencias e diz QUAL celula rodar, em vez de
# estourar NameError longe da causa.
_faltando = [n for n in ('mr', 'REG', 'MODEL_KEY', 'CFG') if n not in globals()]
if _faltando:
    raise RuntimeError(
        'Faltam variaveis: ' + ', '.join(_faltando) + '\n'
        '  mr / REG        -> celula 1 (SETUP)\n'
        '  MODEL_KEY / CFG -> celula 3 (selecionar o modelo)\n'
        'Rode as celulas em ordem a partir da 1.')

import shutil, subprocess, sys


def _gpu():
    """Le a GPU real. Nao assume T4/L4/A100."""
    try:
        out = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=name,memory.total,memory.free,driver_version',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None
    if not out:
        return None
    linha = out.split('\n')[0]
    nome, total, livre, drv = [x.strip() for x in linha.split(',')[:4]]
    return {'name': nome, 'vram_total_gb': float(total) / 1024,
            'vram_free_gb': float(livre) / 1024, 'driver': drv}


GPU = _gpu()
_du = shutil.disk_usage('/content')
DISK_FREE_GB = _du.free / 1024 ** 3
DISK_TOTAL_GB = _du.total / 1024 ** 3

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1024 ** 3
except Exception:
    RAM_GB = None

try:
    import torch
    TORCH_V, CUDA_V = torch.__version__, torch.version.cuda
except Exception:
    TORCH_V = CUDA_V = None

print('AMBIENTE REAL (medido, nao assumido)')
print('  GPU            :', GPU['name'] if GPU else 'NENHUMA')
print('  VRAM total     :', f"{GPU['vram_total_gb']:.1f} GB" if GPU else '-')
print('  VRAM livre     :', f"{GPU['vram_free_gb']:.1f} GB" if GPU else '-')
print('  RAM            :', f'{RAM_GB:.1f} GB' if RAM_GB else 'desconhecida')
print('  Disk total     :', f'{DISK_TOTAL_GB:.1f} GB')
print('  Disk free      :', f'{DISK_FREE_GB:.1f} GB')
print('  CUDA           :', CUDA_V)
print('  Python         :', sys.version.split()[0])
print('  PyTorch        :', TORCH_V)
print()

PF = mr.preflight(MODEL_KEY,
                  available_disk_gb=DISK_FREE_GB,
                  available_vram_gb=GPU['vram_free_gb'] if GPU else None,
                  registry=REG)
print(PF.report())

PREFLIGHT_OK = PF.ready
if not PF.ready:
    # Parar e o comportamento correto. O requisito do modelo NAO e reduzido
    # para caber no runtime.
    raise SystemExit(
        'PARE: ' + PF.status + '\n'
        'Nenhum download foi iniciado e nada foi alterado.\n'
        'Opcoes: escolher um modelo menor (celula 3), trocar de runtime '
        '(Runtime > Change runtime type) ou rodar o cleanup.')

if mr.needs_confirmation(MODEL_KEY, REG):
    print()
    print('!' * 64)
    print('CONFIRMACAO NECESSARIA')
    print(f"  download estimado : {CFG['download_gb']} GB")
    if CFG.get('requires_custom_node'):
        print(f"  custom node       : {CFG['requires_custom_node']}")
        print('  ' + ' '.join(CFG['custom_node_note'].split()))
    if CFG.get('commercial_status') == 'research_only':
        print('  ' + CFG['commercial_banner'])
    print()
    print('Prossiga para a celula de confirmacao.')
    print('!' * 64)
    CONFIRMADO = False
else:
    CONFIRMADO = True


## 6 — Confirmar o download

In [ ]:
# Autoriza o download do modelo selecionado.
if not globals().get('PREFLIGHT_OK'):
    raise RuntimeError(
        'Preflight nao aprovado ou nao executado. Rode a celula de '
        'preflight antes desta.')

CONFIRMADO = True

if not CONFIRMADO:
    raise SystemExit('Download nao autorizado. Nada foi baixado.')
if CFG.get('requires_custom_node'):
    print('Autorizado instalar o custom node', CFG['requires_custom_node'],
          'de', CFG['custom_node_repo'])
print('Autorizado baixar:', MODEL_LABEL)


## 7 — Baixar apenas o modelo selecionado

In [ ]:
# Baixa APENAS o modelo selecionado. Repos e arquivos vem do registry.
from huggingface_hub import hf_hub_download, snapshot_download
import hashlib, pathlib, time

if not PF.ready or not CONFIRMADO:
    raise SystemExit('preflight nao aprovado ou download nao confirmado.')

DEST = pathlib.Path('/content/models') / MODEL_KEY
DEST.mkdir(parents=True, exist_ok=True)
t0 = time.time()

baixados = []
if CFG.get('file'):
    p = hf_hub_download(repo_id=CFG['repo'], filename=CFG['file'],
                        revision=CFG.get('revision'), local_dir=str(DEST))
    baixados.append(pathlib.Path(p))
else:
    d = snapshot_download(repo_id=CFG['repo'], revision=CFG.get('revision'),
                          local_dir=str(DEST))
    baixados = [q for q in pathlib.Path(d).rglob('*')
                if q.is_file() and q.suffix in {'.safetensors', '.gguf'}]

MODEL_RECORD = {
    'model_key': MODEL_KEY,
    'label': MODEL_LABEL,
    'repo': CFG['repo'],
    'revision_requested': CFG.get('revision'),
    'revision_verified': CFG.get('revision_verified', False),
    'license': CFG.get('license'),
    'license_verified': CFG.get('license_verified'),
    'commercial_status': CFG.get('commercial_status'),
    'third_party_quantization': CFG.get('third_party_quantization', False),
    'quantization_license': CFG.get('quantization_license'),
    'quantization_author': CFG.get('quantization_author'),
    'download_seconds': round(time.time() - t0, 1),
    'files': [],
}
for f in baixados:
    h = hashlib.sha256()
    with open(f, 'rb') as fh:
        for bloco in iter(lambda: fh.read(1 << 22), b''):
            h.update(bloco)
    MODEL_RECORD['files'].append(
        {'name': f.name, 'bytes': f.stat().st_size, 'sha256': h.hexdigest()})
    print(f'  {f.name}  {f.stat().st_size / 1e9:.2f} GB  {h.hexdigest()[:16]}...')

print()
print('baixado em', MODEL_RECORD['download_seconds'], 's')
print('AVISO: sha256 acima e do arquivo COMO BAIXADO. O registry nao tinha')
print('hash previo para conferir, entao isto e registro, nao verificacao.')


## 8 — Executar (uma vez)

```
PERSONAGEM ORIGINAL  →  MODELO SELECIONADO  →  OUTPUT
```

Uma execução por vez, por instrução. Sem bateria automática.

In [ ]:
import time, json, pathlib, hashlib

RUN_DIR = mr.run_dir_for(mr.STAGE_MODEL_ONLY, MODEL_KEY,
                         pathlib.Path('/content/ChibiCreate'), REG)
print('run em', RUN_DIR)

RECIPE = {
    'stage': mr.STAGE_MODEL_ONLY,
    'pipeline': 'original -> model',
    'model_key': MODEL_KEY,
    'label': MODEL_LABEL,
    'pipeline_type': CFG['pipeline_type'],
    'input_mode': CFG['input_mode'],
    'model': MODEL_RECORD,
    'commercial_status': CFG['commercial_status'],
    'excluded_from_commercial_ranking': CFG.get(
        'excluded_from_commercial_ranking', False),
    'prompt': PROMPT,
    'negative_prompt': NEGATIVE_PROMPT,
    'prompt_override_applied': PROMPT_INFO['override_applied'],
    'seed': SEED,
    'steps': PARAMS['steps'],
    'cfg': PARAMS['cfg'],
    'sampler': PARAMS['sampler'],
    'scheduler': PARAMS['scheduler'],
    'denoise': PARAMS.get('denoise'),
    'denoise_status': PARAMS.get('denoise_status'),
    'resolution': PARAMS['resolution'],
    'batch': PARAMS['batch'],
    'references': PLANO.to_dict(),
    'reference_files': DISPONIVEIS,
    'environment': {
        'gpu': GPU['name'] if GPU else None,
        'vram_total_gb': round(GPU['vram_total_gb'], 2) if GPU else None,
        'cuda': CUDA_V, 'torch': TORCH_V,
        'ram_gb': round(RAM_GB, 1) if RAM_GB else None,
        'infrastructure': 'google_colab',
        'infrastructure_status': 'EXPERIMENTAL_TEMPORARY',
    },
    'limitations': [
        'uma execucao: nao afirma determinismo',
        'requisitos de VRAM/disco vem do model card, nao de medicao nossa',
    ],
    'approval_status': 'experimental',
}
if PLANO.has_dropped:
    RECIPE['limitations'].append(
        'referencias descartadas por limite do modelo: '
        + ', '.join(PLANO.dropped))
if CFG.get('capability_warning'):
    RECIPE['limitations'].append(' '.join(CFG['capability_warning'].split()))

t0 = time.time()
# ------------------------------------------------------------------
# [HUMAN REVIEW REQUIRED] / [TEST REQUIRED]
# A chamada de inferencia depende do pipeline do modelo selecionado
# (diffusers para LongCat/Z-Image, ComfyUI+GGUF para Qwen, SDXL para Pony).
# Nenhuma delas foi executada ou validada contra hardware real: este
# ambiente nao tem GPU. Escrever a chamada aqui sem nunca te-la rodado
# seria inventar codigo que aparenta funcionar.
#
# O que ESTA pronto e validado: registry, dropdown, preflight, plano de
# referencias, layout de saida, recipe e hashes.
# ------------------------------------------------------------------
raise NotImplementedError(
    'Adapter de inferencia ainda nao implementado para ' + MODEL_KEY + '.\n'
    'Motivo: nunca foi executado contra GPU real; nao vou entregar codigo '
    'de inferencia nao testado disfarcado de pronto.\n'
    'Preflight, download, referencias e recipe acima ja funcionam.')


In [ ]:
# Fecha o recipe depois da inferencia.
import hashlib, json, pathlib

OUT = RUN_DIR / 'output.png'
RECIPE['execution_time'] = round(time.time() - t0, 1)
if OUT.is_file():
    from PIL import Image
    RECIPE['artifact_sha256'] = hashlib.sha256(OUT.read_bytes()).hexdigest()
    with Image.open(OUT) as im:
        RECIPE['output_pixel_sha256'] = hashlib.sha256(
            im.convert('RGBA').tobytes()).hexdigest()
    RECIPE['pixel_hash_note'] = (
        'output_pixel_sha256 e o hash dos PIXELS; artifact_sha256 e o hash '
        'do ARQUIVO. Metadados PNG mudam o segundo sem mudar o primeiro.')

(RUN_DIR / 'recipe.json').write_text(
    json.dumps(RECIPE, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(RECIPE, indent=2, ensure_ascii=False)[:1600])


## 9 — Tabela de comparação

In [ ]:
# Tabela final. Os tres eixos ficam VAZIOS: sao preenchidos por humano.
import pathlib, json

base = pathlib.Path('experiments/model_eval/model_only')
linhas = []
for rec_path in sorted(base.rglob('recipe.json')):
    rec = json.loads(rec_path.read_text())
    linhas.append({
        'model': rec.get('label', rec.get('model_key')),
        'time': rec.get('execution_time'),
        'vram': rec.get('environment', {}).get('vram_total_gb'),
        'status': rec.get('commercial_status'),
    })

print(mr.comparison_table(linhas))


## Avaliação — os três eixos

Preencha à mão. **Não há OVERALL** e não é média aritmética.

| Eixo | O que olhar |
|---|---|
| **DESIGN_PRESERVATION** *(eixo principal)* | roupa, capa, ornamentos, acessórios, chifres |
| **IDENTITY** | rosto, cabelo, cores, é a mesma personagem? |
| **STYLE** | silhueta, proporções chibi, leitura em tamanho pequeno |

Percorra o checklist item a item:

roupa · capa · ornamentos · acessórios · chifres · cabelo · rosto ·
silhueta · proporções chibi · preservação das cores ·
**detalhes inventados ou removidos**

*Simplificar é remover detalhe. Redesenhar é trocar o design.* Um chibi mais
limpo não é perda de design; uma capa que virou outra capa é.

**A decisão artística é humana.** O agente não escolhe vencedor, não aprova
Chibi Master e não julga beleza. Um resultado do Pony **nunca** altera o
candidato comercial.


## 10 — Exportar resultados (antes do reset)

In [ ]:
# Exporta os resultados ANTES de qualquer reset.
import shutil, pathlib, json

d = pathlib.Path('experiments/model_eval/model_only')
if not d.exists():
    raise SystemExit('nada a exportar ainda.')

shutil.make_archive('/content/model_only_results', 'zip', d)
z = pathlib.Path('/content/model_only_results.zip')
print('zip:', round(z.stat().st_size / 1e6, 1), 'MB')

from google.colab import files
files.download(str(z))
print()
print('Baixe o arquivo antes de resetar o runtime.')


## 11 — Cleanup opcional

Libera espaço antes de trocar de modelo. **Não apaga `experiments/`.**

In [ ]:
# CLEANUP OPCIONAL — libera espaco antes de trocar de modelo.
# NAO apaga experiments/: os resultados anteriores ficam intactos.
import shutil, pathlib

ALVOS = [
    pathlib.Path('/content/models'),
    pathlib.Path.home() / '.cache' / 'huggingface',
]

for alvo in ALVOS:
    if alvo.exists():
        tam = sum(f.stat().st_size for f in alvo.rglob('*') if f.is_file())
        shutil.rmtree(alvo, ignore_errors=True)
        print(f'  removido {alvo}  ({tam / 1e9:.1f} GB)')
    else:
        print(f'  ausente  {alvo}')

print()
print('experiments/ NAO foi tocado — resultados preservados.')
print('Baixe o ZIP de resultados ANTES de resetar o runtime.')
print('disco livre agora:', round(shutil.disk_usage('/content').free / 1e9, 1), 'GB')


---

**PARE.** Analise o resultado, baixe o ZIP, resete o runtime e escolha outro
modelo no dropdown. Sem bateria automática.

Não é Flow 02. Não produz `master.png`. Nenhum artefato é aprovado.
